# ALMA continuum primary-beam prescriptions

The continuum driver identifies the telescope from antenna metadata. With
`image_params["primary_beam_model"] = "auto"` (default), ALMA 12-m and 7-m
physical apertures use CASA effective diameters 10.7 m and 6.25 m, respectively,
and CASA's obscured-Airy convention. VLA/EVLA retains its physical 25-m Airy
model. This is independent of visibility-weighting options.

Explicit `list_dish_diameters`, `list_blockage_diameters`, and
`primary_beam_max_radius_1ghz` values take precedence. Select
`primary_beam_model="airy"` to retain the physical-aperture model for ALMA.
MFS reference-frequency beams, MVC channel beams, and cache regeneration use the
same evaluator. Explicit overrides describe effective beam parameters and do
not change the antenna metadata.


In [ ]:
import numpy as np
import xarray as xr

from astroviper.processing_functions.imaging.primary_beam.airy_disk import (
    evaluate_primary_beam,
    resolve_continuum_primary_beam,
)

antennas = xr.Dataset(
    {"ANTENNA_DISH_DIAMETER": ("antenna_name", [12.0])},
    coords={"antenna_name": ["DA01"], "telescope_name": ("antenna_name", ["ALMA"])},
)
geometry = {
    "image_size": [128, 128],
    "image_center": [64, 64],
    "cell_size": np.array([-1.0, 1.0]) * np.pi / (180 * 3600),
}
casa = resolve_continuum_primary_beam(geometry, antennas)
physical = resolve_continuum_primary_beam(
    {**geometry, "primary_beam_model": "airy"},
    antennas,
)
assert casa["list_dish_diameters"] == [10.7]
assert physical["list_dish_diameters"] == [12.0]
assert float(antennas.ANTENNA_DISH_DIAMETER[0]) == 12.0

In [ ]:
profiles = {}
for name, params in [("CASA ALMA", casa), ("Physical aperture", physical)]:
    pb_params = {
        key: params[key]
        for key in (
            "list_dish_diameters",
            "list_blockage_diameters",
        )
    }
    pb_params["ipower"] = 2
    beam = evaluate_primary_beam(np.array([100e9]), ["I"], pb_params, params)
    profiles[name] = beam[0, 0, 0, 64, :]
    assert beam[0, 0, 0, 64, 64] == 1.0

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for label, profile in profiles.items():
    ax.plot(np.arange(128) - 64, profile, label=label)
ax.set(xlabel="Offset (arcsec)", ylabel="Power primary beam", title="100 GHz")
ax.legend()
fig.tight_layout()
plt.close(fig)